# PF2e Spell Metadata - автономная multi-label модель

Тестовая версия без второй модели. Одна нейросеть получает описание заклинания и rank и сразу предсказывает весь итоговый набор labels: официальные `traits` и keywords. Labels хранятся в одном общем пространстве, а при decoding разделяются по префиксам `trait:` и `keyword:`.

Модель возвращает до 5 traits и 5 keywords по умолчанию. Это экспериментальная автономная версия: она нужна для сравнения с двухэтапной архитектурой.

In [1]:
# Этап 1. Импорты и конфигурация.
# Комментарий после этапа: модель специально не содержит никаких вызовов
# второй модели; весь prediction path будет находиться в этом notebook.
from __future__ import annotations

import json
import random
import re
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd()
DATA_DIR = ROOT / "spellsdata"
ARTIFACT_DIR = ROOT / "artifacts" / "standalone_model"
SEED = 42
MAX_TEXT_LENGTH = 256
MAX_VOCAB_SIZE = 12000
MIN_KEYWORD_DOCUMENT_FREQUENCY = 3
DEFAULT_TOP_K_KEYWORDS = 5
DEFAULT_TOP_K_TRAITS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"device={DEVICE}, data_dir={DATA_DIR}")

device=cpu, data_dir=d:\fontys\semester3\PF2e-splells-creations\spellsdata


## Этап 2. Dataset и единое label space

Из каждого Foundry JSON извлекаются description, rank, traits и traditions. Keywords строятся из очищенного текста. В отличие от прошлой версии, здесь нет двух output heads: все labels объединяются в один список. Префикс сохраняет тип label и позволяет вернуть результат в удобном JSON-виде.

In [2]:
TOKEN_PATTERN = re.compile(r"[a-z][a-z0-9'-]{2,}")
HTML_PATTERN = re.compile(r"<[^>]+>")
UUID_PATTERN = re.compile(r"@UUID\[[^]]+\](?:\{([^}]+)\})?")
STOP_WORDS = {
    "the", "and", "for", "that", "this", "with", "from", "into", "your", "you",
    "are", "can", "each", "when", "then", "than", "has", "have", "will", "its",
    "not", "one", "only", "once", "all", "any", "use", "used", "using", "spell",
}

def clean_text(value: str) -> str:
    value = HTML_PATTERN.sub(" ", value or "")
    value = UUID_PATTERN.sub(lambda match: match.group(1) or " ", value)
    return " ".join(value.lower().split())

def tokenize(value: str) -> list[str]:
    return [token for token in TOKEN_PATTERN.findall(clean_text(value)) if token not in STOP_WORDS]

def load_spells(data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for path in sorted(data_dir.rglob("*.json")):
        if path.name == "_folders.json":
            continue
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            system = payload.get("system", {})
            text = clean_text(system.get("description", {}).get("value", ""))
            traits_data = system.get("traits", {})
            official_labels = [*traits_data.get("value", []), *traits_data.get("traditions", [])]
            traits = sorted({str(label).casefold() for label in official_labels if label})
            if text and traits:
                rows.append({
                    "name": payload.get("name", path.stem),
                    "text": text,
                    "tokens": tokenize(text),
                    "rank": int(system.get("level", {}).get("value", 0) or 0),
                    "traits": traits,
                })
        except (OSError, json.JSONDecodeError, TypeError, ValueError) as error:
            print(f"Skipped {path}: {error}")
    return pd.DataFrame(rows)

df = load_spells(DATA_DIR)
document_frequency = Counter(token for tokens in df["tokens"] for token in set(tokens))
keyword_labels = sorted(token for token, count in document_frequency.items()
                        if count >= MIN_KEYWORD_DOCUMENT_FREQUENCY)
keyword_set = set(keyword_labels)
df["keywords"] = df["tokens"].apply(lambda tokens: sorted(set(tokens) & keyword_set))
df = df[df["keywords"].map(bool)].reset_index(drop=True)
trait_labels = sorted({trait for labels in df["traits"] for trait in labels})
all_labels = [f"keyword:{label}" for label in keyword_labels] + [f"trait:{label}" for label in trait_labels]
label_to_id = {label: index for index, label in enumerate(all_labels)}

print(f"usable spells={len(df)}")
print(f"single label space={len(all_labels)} ({len(keyword_labels)} keywords + {len(trait_labels)} traits)")
print(df[["name", "rank", "keywords", "traits"]].head(3).to_string(index=False))

usable spells=1920
single label space=3805 (3705 keywords + 100 traits)
                     name  rank                                                                                                                                                                                                                                                                          keywords                                           traits
        Aberrant Whispers     3 [action, additional, attempt, casting, critical, emanation's, failure, feet, heightened, immune, increase, increases, initial, minds, minute, must, nearby, radius, regardless, result, save, stupefied, success, target, temporarily, those, tongue, unaffected, unknown, utter] [auditory, concentrate, focus, mental, sorcerer]
Accelerated Decomposition     6   [attempt, body, choice, clumsy, creature, creature's, critical, damage, debilitation, double, drained, enfeebled, failure, fortitude, full, half, heightened, increases, minute, must, n

## Этап 3. Vocabulary, split и DataLoader

Targets теперь имеют один multi-hot vector размером `len(all_labels)`. Vocabulary строится только по train split. Rank передаётся в модель как отдельный числовой признак.

In [3]:
def split_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    shuffled = frame.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_start = int(len(shuffled) * 0.8)
    valid_start = int(len(shuffled) * 0.9)
    return shuffled.iloc[:test_start], shuffled.iloc[test_start:valid_start], shuffled.iloc[valid_start:]

train_df, valid_df, test_df = split_frame(df)
train_token_frequency = Counter(token for tokens in train_df["tokens"] for token in tokens)
vocabulary_tokens = [token for token, _ in train_token_frequency.most_common(MAX_VOCAB_SIZE - 2)]
token_to_id = {"<PAD>": 0, "<UNK>": 1, **{token: index + 2 for index, token in enumerate(vocabulary_tokens)}}

def row_labels(row: pd.Series) -> list[str]:
    return [*(f"keyword:{label}" for label in row["keywords"]), *(f"trait:{label}" for label in row["traits"])]

def multi_hot(labels: list[str]) -> Tensor:
    vector = torch.zeros(len(all_labels), dtype=torch.float32)
    for label in labels:
        vector[label_to_id[label]] = 1.0
    return vector

class SpellDataset(Dataset[dict[str, Tensor]]):
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, Tensor]:
        row = self.frame.iloc[index]
        ids = [token_to_id.get(token, token_to_id["<UNK>"]) for token in row["tokens"][:MAX_TEXT_LENGTH]]
        ids += [token_to_id["<PAD>"]] * (MAX_TEXT_LENGTH - len(ids))
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "rank": torch.tensor([row["rank"] / 10.0], dtype=torch.float32),
            "labels": multi_hot(row_labels(row)),
        }

train_loader = DataLoader(SpellDataset(train_df), batch_size=32, shuffle=True)
valid_loader = DataLoader(SpellDataset(valid_df), batch_size=64)
test_loader = DataLoader(SpellDataset(test_df), batch_size=64)

positive_counts = torch.tensor([sum(label in row_labels(row) for _, row in train_df.iterrows()) for label in all_labels], dtype=torch.float32)
pos_weight = ((len(train_df) - positive_counts) / positive_counts.clamp_min(1)).clamp(max=20.0).to(DEVICE)
print(f"split sizes: train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")
print(f"vocabulary={len(token_to_id)}, target width={len(all_labels)}")

split sizes: train=1536, valid=192, test=192
vocabulary=8702, target width=3805


## Этап 4. Одна модель для всех labels

Одна сеть кодирует текст, объединяет его с rank и выдаёт один logits vector. Нет второй модели, co-occurrence post-processing или отдельных prediction calls.

In [4]:
class StandaloneSpellTagger(nn.Module):
    def __init__(self, vocab_size: int, label_count: int, embedding_dim: int = 96, hidden_dim: int = 96) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.encoder = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, label_count),
        )

    def forward(self, input_ids: Tensor, rank: Tensor) -> Tensor:
        encoded, _ = self.encoder(self.embedding(input_ids))
        mask = input_ids.ne(0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        return self.classifier(torch.cat([pooled, rank], dim=1))

model = StandaloneSpellTagger(len(token_to_id), len(all_labels)).to(DEVICE)
loss_function = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
print(f"parameters={sum(parameter.numel() for parameter in model.parameters()):,}")

parameters=1,462,813


## Этап 5. Обучение, evaluation и standalone inference

Threshold применяется после sigmoid, а top-k отдельно ограничивает traits и keywords. Один вызов `predict_spell` получает все labels непосредственно из одной модели.

In [5]:
def move_batch(batch: dict[str, Tensor]) -> dict[str, Tensor]:
    return {key: value.to(DEVICE) for key, value in batch.items()}

def run_epoch(loader: DataLoader[dict[str, Tensor]], training: bool) -> float:
    model.train(training)
    total_loss = 0.0
    for raw_batch in loader:
        batch = move_batch(raw_batch)
        with torch.set_grad_enabled(training):
            logits = model(batch["input_ids"], batch["rank"])
            loss = loss_function(logits, batch["labels"])
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        total_loss += loss.item() * batch["input_ids"].size(0)
    return total_loss / len(loader.dataset)

best_valid_loss = float("inf")
for epoch in range(1, 16):
    train_loss = run_epoch(train_loader, training=True)
    valid_loss = run_epoch(valid_loader, training=False)
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), ARTIFACT_DIR / "model.pt")
    print(f"epoch={epoch:02d} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f}")

model.load_state_dict(torch.load(ARTIFACT_DIR / "model.pt", map_location=DEVICE, weights_only=True))

@torch.no_grad()
def collect_predictions(loader: DataLoader[dict[str, Tensor]]) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    probabilities: list[np.ndarray] = []
    targets: list[np.ndarray] = []
    for raw_batch in loader:
        batch = move_batch(raw_batch)
        probabilities.append(model(batch["input_ids"], batch["rank"]).sigmoid().cpu().numpy())
        targets.append(batch["labels"].cpu().numpy())
    return np.concatenate(probabilities), np.concatenate(targets)

def multilabel_metrics(probabilities: np.ndarray, targets: np.ndarray, threshold: float = 0.35) -> dict[str, float]:
    predicted = probabilities >= threshold
    tp = np.logical_and(predicted, targets == 1).sum()
    fp = np.logical_and(predicted, targets == 0).sum()
    fn = np.logical_and(~predicted, targets == 1).sum()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    micro_f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    per_label_f1: list[float] = []
    for column in range(targets.shape[1]):
        column_tp = np.logical_and(predicted[:, column], targets[:, column] == 1).sum()
        column_fp = np.logical_and(predicted[:, column], targets[:, column] == 0).sum()
        column_fn = np.logical_and(~predicted[:, column], targets[:, column] == 1).sum()
        column_precision = column_tp / max(column_tp + column_fp, 1)
        column_recall = column_tp / max(column_tp + column_fn, 1)
        per_label_f1.append(2 * column_precision * column_recall / max(column_precision + column_recall, 1e-8))
    return {"micro_f1": float(micro_f1), "macro_f1": float(np.mean(per_label_f1)),
            "subset_accuracy": float(np.all(predicted == targets, axis=1).mean())}

test_probabilities, test_targets = collect_predictions(test_loader)
metrics = multilabel_metrics(test_probabilities, test_targets)
print(json.dumps(metrics, indent=2))

@torch.no_grad()
def predict_probabilities(text: str, rank: int) -> Tensor:
    tokens = tokenize(text)[:MAX_TEXT_LENGTH]
    ids = [token_to_id.get(token, token_to_id["<UNK>"]) for token in tokens]
    ids += [token_to_id["<PAD>"]] * (MAX_TEXT_LENGTH - len(ids))
    input_ids = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    rank_tensor = torch.tensor([[rank / 10.0]], dtype=torch.float32, device=DEVICE)
    return model(input_ids, rank_tensor).sigmoid().squeeze(0).cpu()

def decode_group(probabilities: Tensor, prefix: str, threshold: float, top_k: int) -> list[dict[str, float | str]]:
    candidates = []
    for index, label in enumerate(all_labels):
        if label.startswith(prefix) and probabilities[index] >= threshold:
            candidates.append((label.removeprefix(prefix), float(probabilities[index])))
    candidates.sort(key=lambda item: item[1], reverse=True)
    return [{"label": label, "probability": round(probability, 4)} for label, probability in candidates[:max(0, top_k)]]

def predict_spell(text: str, rank: int, top_k_keywords: int = DEFAULT_TOP_K_KEYWORDS,
                  top_k_traits: int = DEFAULT_TOP_K_TRAITS, threshold: float = 0.35) -> dict[str, Any]:
    probabilities = predict_probabilities(text, rank)
    return {
        "text": text,
        "rank": rank,
        "keywords": decode_group(probabilities, "keyword:", threshold, top_k_keywords),
        "traits": decode_group(probabilities, "trait:", threshold, top_k_traits),
        "probabilities": {label: round(float(probabilities[index]), 6) for index, label in enumerate(all_labels)},
        "model_version": "standalone-pytorch-multilabel-v1",
    }

metadata = {"token_to_id": token_to_id, "all_labels": all_labels, "max_text_length": MAX_TEXT_LENGTH,
            "default_top_k_keywords": DEFAULT_TOP_K_KEYWORDS, "default_top_k_traits": DEFAULT_TOP_K_TRAITS,
            "seed": SEED, "model_version": "standalone-pytorch-multilabel-v1"}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
example = predict_spell("A burst of molten fire fills a 30-foot area. Creatures attempt a Reflex save and take persistent fire damage.", rank=3)
print(json.dumps({key: value for key, value in example.items() if key != "probabilities"}, indent=2))
# Комментарий после этапа: этот JSON полностью сформирован одной моделью.
# Вторая модель или co-occurrence обработка не используются.


epoch=01 train_loss=0.4952 valid_loss=0.4030
epoch=02 train_loss=0.4023 valid_loss=0.3976
epoch=03 train_loss=0.3942 valid_loss=0.3930
epoch=04 train_loss=0.3815 valid_loss=0.3830
epoch=05 train_loss=0.3669 valid_loss=0.3730
epoch=06 train_loss=0.3520 valid_loss=0.3657
epoch=07 train_loss=0.3383 valid_loss=0.3618
epoch=08 train_loss=0.3270 valid_loss=0.3582
epoch=09 train_loss=0.3159 valid_loss=0.3563
epoch=10 train_loss=0.3039 valid_loss=0.3567
epoch=11 train_loss=0.2928 valid_loss=0.3566
epoch=12 train_loss=0.2813 valid_loss=0.3563
epoch=13 train_loss=0.2693 valid_loss=0.3560
epoch=14 train_loss=0.2578 valid_loss=0.3612
epoch=15 train_loss=0.2464 valid_loss=0.3647
{
  "micro_f1": 0.18512048279745316,
  "macro_f1": 0.05104890994245868,
  "subset_accuracy": 0.0
}
{
  "text": "A burst of molten fire fills a 30-foot area. Creatures attempt a Reflex save and take persistent fire damage.",
  "rank": 3,
  "keywords": [
    {
      "label": "area",
      "probability": 0.9922
    },
    {
  